In [2]:
import os
import sys
import json
from pathlib import Path
from typing import Any, Dict, List, Optional
import logging

import numpy as np
import pandas as pd
from sqlalchemy.orm import Session

# konfiguracja logowania w notebooku
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("dataset-notebook")

# ustawienie ścieżki do backendu, żeby móc importować moduły app.*
PROJECT_ROOT = Path("..").resolve()
BACKEND_PATH = PROJECT_ROOT / "backend"
if str(BACKEND_PATH) not in sys.path:
    sys.path.insert(0, str(BACKEND_PATH))

from app.database import SessionLocal
from app.models.user import User
from app.models.symptom_profile import SymptomProfile
from app.services.embedding_service import generate_embedding, generate_embeddings_batch
from app.services.matching_service import find_matching_group, add_user_to_group
from app.routers.auth import _hash_password

DATA_PATH = PROJECT_ROOT / "data" / "test_data.json"
DRY_RUN = False

/home/mario/miniconda3/envs/zebrapoint/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from contextlib import contextmanager
from collections import Counter


@contextmanager
def get_db() -> Session:
    """Kontekst menedżer na sesję DB (tak jak w backendzie, ale dla notebooka)."""
    db = SessionLocal()
    try:
        yield db
        db.commit()
    except Exception:
        db.rollback()
        raise
    finally:
        db.close()


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    """Wczytuje testowy zbiór danych z JSON-a."""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Plik JSON powinien zawierać listę obiektów")

    records: List[Dict[str, Any]] = []
    for idx, raw in enumerate(data, start=1):
        email = raw.get("email")
        password = raw.get("haslo")
        description = raw.get("opis")
        display_name = raw.get("display_name")

        if not email or not password or not description:
            raise ValueError(f"Rekord {idx} ma brakujące pola (email/haslo/opis)")

        # display_name jest wymagany zgodnie z planem; jeśli go nie ma,
        # można doraźnie użyć prefiksu z emaila.
        if not display_name:
            display_name = email.split("@")[0]

        description = str(description).strip()
        if len(description) < 100:
            raise ValueError(
                f"Opis w rekordzie {idx} jest za krótki ({len(description)} znaków). "
                "Potrzebne >= 100 znaków."
            )

        records.append(
            {
                "email": str(email).strip(),
                "password": str(password),
                "display_name": str(display_name).strip(),
                "description": description,
            }
        )

    logger.info("Wczytano %d rekordów z %s", len(records), path)
    return records


def create_user_from_record(db: Session, record: Dict[str, Any]) -> User:
    """Tworzy użytkownika na podstawie rekordu z JSON-a albo zwraca istniejącego."""
    email = record["email"]
    display_name = record["display_name"]
    password = record["password"]

    user = db.query(User).filter(User.email == email).first()
    if user:
        logger.info("Użytkownik %s już istnieje — pomijam tworzenie", email)
        return user

    user = User(
        email=email,
        password_hash=_hash_password(password),
        display_name=display_name,
    )
    db.add(user)
    db.flush()
    logger.info("Utworzono użytkownika %s", email)
    return user


def create_or_update_symptom_profile(
    db: Session,
    user: User,
    description: str,
) -> SymptomProfile:
    """Tworzy lub aktualizuje profil objawów i dopasowuje grupę."""
    profile = (
        db.query(SymptomProfile)
        .filter(SymptomProfile.user_id == user.id)
        .first()
    )

    embedding = generate_embedding(description)
    match = find_matching_group(db, embedding, user.id)
    group_id = match["group_id"]

    if profile is None:
        profile = SymptomProfile(
            user_id=user.id,
            description=description,
            embedding=embedding,
            group_id=group_id,
            match_score=match["score"],
        )
        db.add(profile)
        logger.info("Utworzono profil objawów dla %s", user.email)
    else:
        profile.description = description
        profile.embedding = embedding
        profile.group_id = group_id
        profile.match_score = match["score"]
        logger.info("Zaktualizowano profil objawów dla %s", user.email)

    add_user_to_group(db, user.id, group_id)

    db.flush()
    return profile

In [4]:
# Test połączenia z bazą Supabase

with get_db() as db:
    try:
        user_count = db.query(User).count()
        print(f"Połączenie OK. Liczba użytkowników w bazie: {user_count}")
    except Exception as exc:
        print("Błąd połączenia z bazą:", exc)
        raise

2026-03-12 18:12:30,865 INFO sqlalchemy.engine.Engine select pg_catalog.version()


INFO:sqlalchemy.engine.Engine:select pg_catalog.version()


2026-03-12 18:12:30,866 INFO sqlalchemy.engine.Engine [raw sql] {}


INFO:sqlalchemy.engine.Engine:[raw sql] {}


2026-03-12 18:12:30,927 INFO sqlalchemy.engine.Engine select current_schema()


...

INFO:sqlalchemy.engine.Engine:SELECT count(*) AS count_1 
FROM (SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users) AS anon_1


2026-03-12 18:12:31,051 INFO sqlalchemy.engine.Engine [generated in 0.00090s] {}


INFO:sqlalchemy.engine.Engine:[generated in 0.00090s] {}


Połączenie OK. Liczba użytkowników w bazie: 4
2026-03-12 18:12:31,124 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


In [8]:
# Wczytanie danych z JSON i podgląd struktury

records = load_dataset(DATA_PATH)
print(f"Liczba rekordów w zbiorze: {len(records)}")

df_preview = pd.DataFrame(records)
df_preview.head()

INFO:dataset-notebook:Wczytano 27 rekordów z /home/mario/workspace/zebrapoint/data/test_data.json


Liczba rekordów w zbiorze: 27


,email,password,display_name,description
0,test0004@zebra.pl,test1234,Mama 4,Nasze dziecko urodziło się z bardzo niską masą...
1,test0005@zebra.pl,test1234,Mama 5,Od urodzenia zauważyliśmy u córki bardzo jasną...
2,test0006@zebra.pl,test1234,Mama 6,Syn od pierwszych miesięcy życia miał trudnośc...
3,test0007@zebra.pl,test1234,Mama 7,Nasze dziecko od małego bardzo mało mówiło. Te...
4,test0008@zebra.pl,test1234,Mama 8,Córka od niemowlęctwa miała częste napady drga...


In [9]:
# Utworzenie użytkowników w tabeli `users` na podstawie rekordów

created = 0
existing = 0

with get_db() as db:
    for rec in records:
        user_before = db.query(User).filter(User.email == rec["email"]).first()
        user = create_user_from_record(db, rec)
        if user_before is None:
            created += 1
        else:
            existing += 1

print(f"Nowo utworzonych użytkowników: {created}")
print(f"Użytkownicy już istniejący:   {existing}")

2026-03-12 18:23:01,813 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-12 18:23:01,818 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


INFO:sqlalchemy.engine.Engine:SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


2026-03-12 18:23:01,819 INFO sqlalchemy.engine.Engine [generated in 0.00129s] {'email_1': 'test0004@zebra.pl', 'param_1': 1}


...

2026-03-12 18:23:10,972 INFO sqlalchemy.engine.Engine [cached since 8.725s ago] {'id': UUID('a0be9ef9-5920-45ca-8cef-769a8e12ec70'), 'email': 'test0030@zebra.pl', 'password_hash': '$2b$12$xIqusIwkecfIwjWLGp6fZurDLPrTUTLFkbDmBzNH9WUuDk6fPWc5K', 'display_name': 'Mama 30', 'avatar_url': None, 'is_active': True}


INFO:sqlalchemy.engine.Engine:[cached since 8.725s ago] {'id': UUID('a0be9ef9-5920-45ca-8cef-769a8e12ec70'), 'email': 'test0030@zebra.pl', 'password_hash': '$2b$12$xIqusIwkecfIwjWLGp6fZurDLPrTUTLFkbDmBzNH9WUuDk6fPWc5K', 'display_name': 'Mama 30', 'avatar_url': None, 'is_active': True}
INFO:dataset-notebook:Utworzono użytkownika test0030@zebra.pl


2026-03-12 18:23:11,002 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Nowo utworzonych użytkowników: 27
Użytkownicy już istniejący:   0


In [10]:
# Utworzenie / aktualizacja profili objawów i dopasowanie do grup

profiles_created = 0
profiles_updated = 0

with get_db() as db:
    for rec in records:
        email = rec["email"]
        user = db.query(User).filter(User.email == email).first()
        if user is None:
            logger.warning("Brak użytkownika %s — pomijam profil objawów", email)
            continue

        existing_profile = (
            db.query(SymptomProfile)
            .filter(SymptomProfile.user_id == user.id)
            .first()
        )

        profile = create_or_update_symptom_profile(db, user, rec["description"])

        if existing_profile is None:
            profiles_created += 1
        else:
            profiles_updated += 1

print(f"Nowo utworzonych profili objawów: {profiles_created}")
print(f"Zaktualizowanych profili objawów: {profiles_updated}")

2026-03-12 18:23:45,785 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-12 18:23:45,787 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


INFO:sqlalchemy.engine.Engine:SELECT users.id AS users_id, users.email AS users_email, users.password_hash AS users_password_hash, users.display_name AS users_display_name, users.avatar_url AS users_avatar_url, users.is_active AS users_is_active, users.created_at AS users_created_at, users.updated_at AS users_updated_at 
FROM users 
WHERE users.email = %(email_1)s 
 LIMIT %(param_1)s


2026-03-12 18:23:45,788 INFO sqlalchemy.engine.Engine [cached since 43.97s ago] {'email_1': 'test0004@zebra.pl', 'param_1': 1}


...

2026-03-12 18:23:58,882 INFO sqlalchemy.engine.Engine [cached since 7.897s ago] {'id': UUID('cfa7d648-9257-4254-877b-d0b3af07a294'), 'user_id': UUID('a0be9ef9-5920-45ca-8cef-769a8e12ec70'), 'description': 'Córka rozwijała się dobrze przez pierwsze lata, ale w ostatnim czasie zauważyliśmy pogorszenie pamięci i koncentracji. Zaczęła też mieć trudności z mówieniem.', 'embedding': '[0.06316517293453217,-0.001949384342879057,0.03590528666973114,0.057732272893190384,-0.05324122682213783,0.0032863644883036613,-0.04048977792263031,0 ... (7742 characters truncated) ... 58,0.02689574472606182,-0.050712306052446365,0.03528808429837227,0.03273146599531174,0.07506851851940155,0.0029041199013590813,-0.007419778499752283]', 'group_id': '29938bd3-d18f-49a2-83c1-697340b25a31', 'match_score': 0.747617207396552}


INFO:sqlalchemy.engine.Engine:[cached since 7.897s ago] {'id': UUID('cfa7d648-9257-4254-877b-d0b3af07a294'), 'user_id': UUID('a0be9ef9-5920-45ca-8cef-769a8e12ec70'), 'description': 'Córka rozwijała się dobrze przez pierwsze lata, ale w ostatnim czasie zauważyliśmy pogorszenie pamięci i koncentracji. Zaczęła też mieć trudności z mówieniem.', 'embedding': '[0.06316517293453217,-0.001949384342879057,0.03590528666973114,0.057732272893190384,-0.05324122682213783,0.0032863644883036613,-0.04048977792263031,0 ... (7742 characters truncated) ... 58,0.02689574472606182,-0.050712306052446365,0.03528808429837227,0.03273146599531174,0.07506851851940155,0.0029041199013590813,-0.007419778499752283]', 'group_id': '29938bd3-d18f-49a2-83c1-697340b25a31', 'match_score': 0.747617207396552}


2026-03-12 18:23:58,913 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT


Nowo utworzonych profili objawów: 27
Zaktualizowanych profili objawów: 0


In [12]:
# Diagnostyka: liczba profili i rozkład po grupach

with get_db() as db:
    total_profiles = db.query(SymptomProfile).count()
    print(f"Liczba profili objawów w bazie: {total_profiles}")

    rows = (
        db.query(SymptomProfile.group_id)
        .all()
    )


group_ids = [row[0] for row in rows]
counts = Counter(group_ids)

print("\nRozkład liczby profili w grupach:")
for group_id, cnt in counts.items():
    print(f"  grupa={group_id}: {cnt} profili")

2026-03-12 18:24:56,359 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-03-12 18:24:56,361 INFO sqlalchemy.engine.Engine SELECT count(*) AS count_1 
FROM (SELECT symptom_profiles.id AS symptom_profiles_id, symptom_profiles.user_id AS symptom_profiles_user_id, symptom_profiles.description AS symptom_profiles_description, symptom_profiles.embedding AS symptom_profiles_embedding, symptom_profiles.group_id AS symptom_profiles_group_id, symptom_profiles.match_score AS symptom_profiles_match_score, symptom_profiles.created_at AS symptom_profiles_created_at, symptom_profiles.updated_at AS symptom_profiles_updated_at 
FROM symptom_profiles) AS anon_1


INFO:sqlalchemy.engine.Engine:SELECT count(*) AS count_1 
FROM (SELECT symptom_profiles.id AS symptom_profiles_id, symptom_profiles.user_id AS symptom_profiles_user_id, symptom_profiles.description AS symptom_profiles_description, symptom_profiles.embedding AS symptom_profiles_embedding, symptom_profiles.group_id AS symptom_profiles_group_id, symptom_profiles.match_score AS symptom_profiles_match_score, symptom_profiles.created_at AS symptom_profiles_created_at, symptom_profiles.updated_at AS symptom_profiles_updated_at 
FROM symptom_profiles) AS anon_1


2026-03-12 18:24:56,362 INFO sqlalchemy.engine.Engine [cached since 12.81s ago] {}


...

2026-03-12 18:24:56,419 INFO sqlalchemy.engine.Engine [cached since 12.81s ago] {}


INFO:sqlalchemy.engine.Engine:[cached since 12.81s ago] {}


2026-03-12 18:24:56,446 INFO sqlalchemy.engine.Engine COMMIT


INFO:sqlalchemy.engine.Engine:COMMIT



Rozkład liczby profili w grupach:
  grupa=66204af3-89e7-4ec6-838e-0fcd7e54e4c7: 2 profili
  grupa=2d2a759b-f914-4b1f-925e-f11e72c3585a: 10 profili
  grupa=cb092a24-2831-48f5-b35f-7add54e19d6b: 4 profili
  grupa=13b24b56-7537-41b2-b484-fa811047b471: 1 profili
  grupa=81673b39-a3a1-4a66-8e03-c6ace83cdf6b: 1 profili
  grupa=52430a0b-05ed-41fb-8c80-6390c23cd7bd: 2 profili
  grupa=d08e312f-e27d-4546-ad2b-aee50f883b60: 1 profili
  grupa=55c850c8-071d-4f86-a619-16e010387f42: 1 profili
  grupa=708a6370-ea74-4374-a698-5adad01af45a: 1 profili
  grupa=75f17305-fcd8-4ee6-a2bf-d80329866d08: 1 profili
  grupa=29938bd3-d18f-49a2-83c1-697340b25a31: 2 profili
  grupa=da9964cc-5ab7-40e8-8d34-dce853914c8f: 1 profili
  grupa=88e3bb48-85f0-4605-88f5-2253111c0785: 1 profili
  grupa=3d2f5bd7-90e7-4062-bb72-108b9c7f61bb: 1 profili
  grupa=7836116a-3678-4346-b347-f1319d70885b: 1 profili
  grupa=6c19d249-d587-4ae0-a7cb-216551c5d268: 1 profili


In [13]:
# Eksperyment ML: prosta klasteryzacja KMeans na embeddingach

from sklearn.cluster import KMeans

# Weźmy embeddingi dla opisów z JSON-a (bez czytania z bazy)
texts = [rec["description"] for rec in records]
embeddings = generate_embeddings_batch(texts)

X = np.array(embeddings)
print("Macierz embeddingów:", X.shape)

# Przy niewielkiej liczbie rekordów spróbujmy np. 3 klastry
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
cluster_labels = kmeans.fit_predict(X)

# Podgląd wyników w DataFrame
ml_df = pd.DataFrame(
    {
        "email": [rec["email"] for rec in records],
        "cluster": cluster_labels,
    }
)

display(ml_df.head())
print("\nLiczność klastrów:")
print(ml_df["cluster"].value_counts())

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.54it/s]

Macierz embeddingów: (27, 384)


,email,cluster
0,test0004@zebra.pl,0
1,test0005@zebra.pl,2
2,test0006@zebra.pl,0
3,test0007@zebra.pl,1
4,test0008@zebra.pl,0



Liczność klastrów:
cluster
0    14
1     9
2     4
Name: count, dtype: int64


In [ ]:
def run_full_seed(dry_run: bool = False) -> None:
    """Wysokopoziomowa funkcja do ponownego uruchamiania całego seeda.

    Kroki:
    - wczytanie JSON,
    - stworzenie użytkowników,
    - stworzenie / aktualizacja profili objawów i dopasowanie grup.
    """
    global DRY_RUN, records
    DRY_RUN = dry_run

    # wczytaj dane
    records = load_dataset(DATA_PATH)
    logger.info("Start seeda. Rekordów: %d (DRY_RUN=%s)", len(records), DRY_RUN)

    created = 0
    existing = 0
    profiles_created = 0
    profiles_updated = 0

    with get_db() as db:
        for rec in records:
            # użytkownik
            user_before = db.query(User).filter(User.email == rec["email"]).first()
            user = create_user_from_record(db, rec)
            if user_before is None:
                created += 1
            else:
                existing += 1

            # profil objawów
            existing_profile = (
                db.query(SymptomProfile)
                .filter(SymptomProfile.user_id == user.id)
                .first()
            )
            profile = create_or_update_symptom_profile(db, user, rec["description"])
            if existing_profile is None:
                profiles_created += 1
            else:
                profiles_updated += 1

        if DRY_RUN:
            logger.info("DRY_RUN=True — wykonuję rollback zmian")
            db.rollback()
        else:
            logger.info("Zatwierdzam zmiany w bazie")

    print("Seed zakończony:\n")
    print(f"  Nowo utworzonych użytkowników: {created}")
    print(f"  Użytkownicy już istniejący:   {existing}")
    print(f"  Nowo utworzonych profili:     {profiles_created}")
    print(f"  Zaktualizowanych profili:     {profiles_updated}")


# Przykład użycia:
# run_full_seed(dry_run=True)  # test bez zapisywania do bazy
# run_full_seed(dry_run=False) # realny seed do bazy